In [1]:
import pandas as pd
import numpy as np
from itertools import product
import plotly.graph_objects as go
import nbformat

# Load the Excel file
file_path = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey diagram table.xlsx"
df = pd.read_excel(file_path, sheet_name='RAW')

print("Original data shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

Original data shape: (56, 4)

First few rows:
                                    selected article  \
0  Explainable AI Meets MobileNetV2: A Multi-Bran...   
1  Deep learning technique for plant disease clas...   
2  OptiNet-B3: a lightweight explainable deep lea...   
3  Ensemble transfer learning meets explainable A...   
4  DBA-ViNet: an effective deep learning framewor...   

                                 Crops Name  \
0                                     Apple   
1       Apple, Apricot, Peach, Pear, Walnut   
2        Apple, Corn, Grape, Potato, Tomato   
3        Apple, Corn, Grape, Potato, Tomato   
4  Apple, Guava, Mango, Orange, Pomegranate   

                                     Model Type Used  \
0  Multi-Branched MobileNetV2, ResNet, DenseNet, ...   
1                                           ResNet-9   
2                       OptiNet-B3 (Lightweight CNN)   
3  LITENET (MobileNetV3 + EfficientNetV2), DenseN...   
4  DBA-ViNet, EfficientNetV2, ConvNeXt, YOLOv8, M...  

In [2]:
# Function to clean and split comma-separated values
def split_and_clean(value):
    """Split comma-separated values and strip whitespace"""
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(',') if item.strip()]

# Expand the data: create all combinations per article
expanded_records = []

for idx, row in df.iterrows():
    article = row['selected article']
    crops = split_and_clean(row['Crops Name'])
    models = split_and_clean(row['Model Type Used'])
    xai_methods = split_and_clean(row['XAI Methods Used'])
    
    # Create all combinations (cartesian product)
    for crop, model, xai in product(crops, models, xai_methods):
        expanded_records.append({
            'Crops Name': crop,
            'Model Type Used': model,
            'XAI Methods Used': xai,
            'article_count': 1  # Each combination from an article counts as 1
        })

# Create expanded dataframe
expanded_df = pd.DataFrame(expanded_records)

print(f"Total articles: {len(df)}")
print(f"Total expanded records: {len(expanded_df)}")
print(f"\nUnique Crops: {expanded_df['Crops Name'].nunique()}")
print(f"Unique Models: {expanded_df['Model Type Used'].nunique()}")
print(f"Unique XAI Methods: {expanded_df['XAI Methods Used'].nunique()}")
print(f"\nFirst 10 expanded records:")
print(expanded_df.head(10))

# Aggregate counts per flow
flow_counts = expanded_df.groupby(['Crops Name', 'Model Type Used', 'XAI Methods Used']).size().reset_index(name='count')
print(f"\nUnique flows: {len(flow_counts)}")
print(flow_counts.head(10))

Total articles: 56
Total expanded records: 419

Unique Crops: 43
Unique Models: 110
Unique XAI Methods: 23

First 10 expanded records:
  Crops Name             Model Type Used XAI Methods Used  article_count
0      Apple  Multi-Branched MobileNetV2             LIME              1
1      Apple  Multi-Branched MobileNetV2             UMAP              1
2      Apple                      ResNet             LIME              1
3      Apple                      ResNet             UMAP              1
4      Apple                    DenseNet             LIME              1
5      Apple                    DenseNet             UMAP              1
6      Apple                    Xception             LIME              1
7      Apple                    Xception             UMAP              1
8      Apple                    ResNet-9             SHAP              1
9      Apple                    ResNet-9         Grad-CAM              1

Unique flows: 414
  Crops Name                         Model 

In [3]:
# Prepare data for sankey diagram
# Create source and target nodes with counts

# Step 1: Create nodes for each level
crops_nodes = sorted(flow_counts['Crops Name'].unique().tolist())
models_nodes = sorted(flow_counts['Model Type Used'].unique().tolist())
xai_nodes = sorted(flow_counts['XAI Methods Used'].unique().tolist())

# Add labels to distinguish between levels
crops_labeled = [f"{c} ({flow_counts[flow_counts['Crops Name']==c]['count'].sum()})" for c in crops_nodes]
models_labeled = [f"{m} ({flow_counts[flow_counts['Model Type Used']==m]['count'].sum()})" for m in models_nodes]
xai_labeled = [f"{x} ({flow_counts[flow_counts['XAI Methods Used']==x]['count'].sum()})" for x in xai_nodes]

# All nodes in order
all_nodes = crops_labeled + models_labeled + xai_labeled

# Step 2: Create source, target, and value lists for sankey
source_indices = []
target_indices = []
values = []
colors = []

# Create mappings
crop_to_idx = {c: i for i, c in enumerate(crops_labeled)}
model_to_idx = {m: i + len(crops_labeled) for i, m in enumerate(models_labeled)}
xai_to_idx = {x: i + len(crops_labeled) + len(models_labeled) for i, x in enumerate(xai_labeled)}

# Crop to Model links
for _, row in flow_counts.iterrows():
    crop_label = f"{row['Crops Name']} ({flow_counts[flow_counts['Crops Name']==row['Crops Name']]['count'].sum()})"
    model_label = f"{row['Model Type Used']} ({flow_counts[flow_counts['Model Type Used']==row['Model Type Used']]['count'].sum()})"
    
    source_indices.append(crop_to_idx[crop_label])
    target_indices.append(model_to_idx[model_label])
    values.append(row['count'])

# Model to XAI links
for _, row in flow_counts.iterrows():
    model_label = f"{row['Model Type Used']} ({flow_counts[flow_counts['Model Type Used']==row['Model Type Used']]['count'].sum()})"
    xai_label = f"{row['XAI Methods Used']} ({flow_counts[flow_counts['XAI Methods Used']==row['XAI Methods Used']]['count'].sum()})"
    
    source_indices.append(model_to_idx[model_label])
    target_indices.append(xai_to_idx[xai_label])
    values.append(row['count'])

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node = dict(
        pad = 15,
        thickness = 20,
        line = dict(color = 'black', width = 0.5),
        label = all_nodes,
        color = 'blue'
    ),
    link = dict(
        source = source_indices,
        target = target_indices,
        value = values,
        color = 'rgba(200, 200, 200, 0.4)'
    )
)])

fig.update_layout(
    title_text="Sankey Diagram: Crops → Models → XAI Methods (with Record Counts)",
    font_size=10,
    height=800,
    width=1400,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()

# Save the diagram
output_path = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey_diagram.html"
fig.write_html(output_path)
print(f"\nSankey diagram saved at: {output_path}")


Sankey diagram saved at: C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey_diagram.html
